# SimWorld Studio

**AI-powered 3D scene generation platform** — chat with Claude to build urban scenes in real-time.

## What this notebook does
1. Checks your Colab GPU
2. Installs SimWorld (official binary from HuggingFace)
3. Installs SimWorld Studio platform
4. Asks for your Anthropic API key (stays local, never sent to us)
5. Launches everything and gives you a browser URL

**Run all cells in order.** Total setup time: ~5 minutes.

---

## Cell 1: GPU Check

In [ ]:
"""Verify GPU is available — T4 or better required."""
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                       capture_output=True, text=True)
if result.returncode != 0:
    print("ERROR: No GPU detected!")
    print("Go to: Runtime -> Change runtime type -> GPU (T4)")
    sys.exit(1)

gpu_info = result.stdout.strip()
print(f"GPU: {gpu_info}")

supported = ["T4", "A100", "V100", "L4", "A10"]
if not any(g in gpu_info for g in supported):
    print(f"WARNING: Unrecognized GPU. Supported: {supported}")
    print("Continuing anyway...")
else:
    print("GPU check passed!")

## Cell 2: Install SimWorld (Official)

Downloads the SimWorld binary from [HuggingFace](https://huggingface.co/datasets/SimWorld-AI/SimWorld) and installs the Python client from [GitHub](https://github.com/SimWorld-AI/SimWorld).

In [ ]:
"""Install SimWorld: Python client + UE binary from HuggingFace."""
import os

# --- Python client ---
if not os.path.exists("/content/SimWorld/setup.py"):
    print("[1/2] Cloning SimWorld Python client...")
    !git clone --depth 1 https://github.com/SimWorld-AI/SimWorld.git /content/SimWorld
    !cd /content/SimWorld && pip install -q -e .
    print("SimWorld Python client installed.")
else:
    print("[1/2] SimWorld Python client already installed.")

# --- UE binary (Base package) ---
SIMWORLD_BINARY_DIR = "/content/SimWorld-Binary"

def find_simworld_sh(base):
    """Search for SimWorld.sh recursively."""
    for root, dirs, files in os.walk(base):
        if "SimWorld.sh" in files:
            return os.path.join(root, "SimWorld.sh")
    return None

SIMWORLD_SH = find_simworld_sh(SIMWORLD_BINARY_DIR)

if not SIMWORLD_SH:
    print("[2/2] Downloading SimWorld binary from HuggingFace (~3-5GB)...")
    !mkdir -p {SIMWORLD_BINARY_DIR}
    !wget -q --show-progress -O /tmp/simworld_linux.zip \
        https://huggingface.co/datasets/SimWorld-AI/SimWorld/resolve/main/Base/Linux.zip
    !unzip -q /tmp/simworld_linux.zip -d {SIMWORLD_BINARY_DIR}
    !rm -f /tmp/simworld_linux.zip
    SIMWORLD_SH = find_simworld_sh(SIMWORLD_BINARY_DIR)
    if SIMWORLD_SH:
        os.chmod(SIMWORLD_SH, 0o755)
        print(f"SimWorld binary installed at: {SIMWORLD_SH}")
    else:
        print("ERROR: SimWorld.sh not found after extraction!")
        print("Contents:")
        !find {SIMWORLD_BINARY_DIR} -maxdepth 3 -type f | head -20
else:
    print(f"[2/2] SimWorld binary already installed at: {SIMWORLD_SH}")

print(f"\nSimWorld launch script: {SIMWORLD_SH}")

## Cell 3: Install Studio Platform

In [ ]:
"""Install the Studio platform + Claude Code CLI."""
import subprocess, shutil, os

# --- Arena platform (pip package) ---
print("[1/3] Installing SimWorld Studio...")

# Try release URL first, fall back to local build
STUDIO_PKG_URL = "https://github.com/SimWorld-AI/SimWorld-Studio/releases/download/v0.1.0/simworld_studio-0.1.0.tar.gz"
local_pkg = "/content/SimWorld-Studio-Release/dist/simworld_studio-0.1.0.tar.gz"

if os.path.exists(local_pkg):
    !pip install -q {local_pkg}
    print("  Installed from local build.")
else:
    result = subprocess.run(
        ["pip", "install", "-q", STUDIO_PKG_URL],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("  Installed from release URL.")
    else:
        print(f"  ERROR: Could not install package.")
        print(f"  {result.stderr[:200]}")
        print("  Please check the release URL or upload the package manually.")

# Verify installation
studio_installed = shutil.which("simworld-studio") is not None
if studio_installed:
    ver = subprocess.run(["simworld-studio", "version"], capture_output=True, text=True).stdout.strip()
    print(f"  {ver}")
else:
    print("  WARNING: simworld-studio CLI not found after install!")

# --- Node.js (usually pre-installed in Colab) ---
print("[2/3] Checking Node.js...")
if not shutil.which("node"):
    print("  Installing Node.js...")
    !apt-get install -y -qq nodejs npm
else:
    node_ver = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
    print(f"  Node.js {node_ver} found.")

# --- Claude Code CLI ---
print("[3/3] Installing Claude Code CLI...")
if not shutil.which("claude"):
    !npm install -g @anthropic-ai/claude-code 2>/dev/null | tail -2
    print("  Claude Code CLI installed.")
else:
    claude_ver = subprocess.run(["claude", "--version"], capture_output=True, text=True).stdout.strip()
    print(f"  Claude Code CLI {claude_ver} already installed.")

print("\nAll dependencies installed!")

## Cell 4: Enter Your API Key

Your Anthropic API key stays **entirely local** in this Colab runtime. It is **never** sent to SimWorld servers — it's only used by the Claude Code CLI running inside your Colab session.

Get your key at [console.anthropic.com](https://console.anthropic.com)

In [ ]:
"""Enter your Anthropic API key — stays local, never sent anywhere."""
from getpass import getpass
import os

api_key = getpass("Enter your Anthropic API key (sk-ant-...): ")

# Always set the key — warn if format looks unusual
os.environ["ANTHROPIC_API_KEY"] = api_key

if not api_key.startswith("sk-ant-"):
    print("WARNING: Key doesn't start with 'sk-ant-'. Double-check your key.")
    print("Get yours at: https://console.anthropic.com")
    print("Key was still set — continuing.")
else:
    print("API key set! (stored only in this runtime's memory — never transmitted)")

## Cell 5: Launch SimWorld + Studio

In [ ]:
"""Start SimWorld (headless GPU) and the Studio backend."""
import subprocess, time, os, socket

# --- Install headless rendering dependencies ---
print("[1/4] Installing rendering dependencies...")
r = subprocess.run(
    ["apt-get", "install", "-y", "-qq",
     "xvfb", "vulkan-tools", "mesa-vulkan-drivers",
     "libvulkan1", "libegl1", "libgles2", "libgl1"],
    capture_output=True
)
if r.returncode != 0:
    print(f"  WARNING: Some rendering deps may have failed (exit {r.returncode})")
else:
    print("  Rendering deps installed.")

# --- Start virtual display ---
print("[2/4] Starting virtual display...")
subprocess.run(["pkill", "-f", "Xvfb"], capture_output=True)
time.sleep(1)
xvfb_proc = subprocess.Popen(
    ["Xvfb", ":99", "-screen", "0", "1280x720x24"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
os.environ["DISPLAY"] = ":99"
time.sleep(2)
print("  Virtual display :99 started.")

# --- Launch SimWorld headless ---
print("[3/4] Launching SimWorld (this takes 30-60 seconds)...")

# Find SimWorld.sh if not set from Cell 2
if 'SIMWORLD_SH' not in globals() or not globals().get('SIMWORLD_SH'):
    for root, dirs, files in os.walk("/content/SimWorld-Binary"):
        if "SimWorld.sh" in files:
            SIMWORLD_SH = os.path.join(root, "SimWorld.sh")
            break

simworld_dir = os.path.dirname(SIMWORLD_SH)

ue_proc = subprocess.Popen(
    [SIMWORLD_SH,
     "/Game/Maps/empty.umap",
     "-RenderOffScreen", "-Unattended",
     "-NOSPLASH", "-NOSOUND",
     "-ResX=1280", "-ResY=720",
     "-Messaging",
     "-PixelStreamingIP=127.0.0.1", "-PixelStreamingPort=8586"],
    stdout=open("/content/ue.log", "w"),
    stderr=subprocess.STDOUT,
    cwd=simworld_dir
)

# Wait for UE to boot — check TCP port 9000 (SimWorld default)
SIMWORLD_PORT = 9000
print(f"  Waiting for SimWorld to start (checking port {SIMWORLD_PORT})...")
started = False
for attempt in range(60):  # up to 120 seconds
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(2)
        s.connect(("127.0.0.1", SIMWORLD_PORT))
        s.close()
        started = True
        break
    except (ConnectionRefusedError, socket.timeout, OSError):
        time.sleep(2)
        if attempt % 10 == 0 and attempt > 0:
            print(f"  Still waiting... ({attempt*2}s)")

if started:
    print("  SimWorld is running!")
else:
    print("  WARNING: SimWorld may not be fully started yet.")
    print("  Check /content/ue.log for details.")
    print("  Last 10 lines of log:")
    !tail -10 /content/ue.log

# --- Launch Arena platform ---
print("[4/4] Starting Studio backend...")
studio_proc = subprocess.Popen(
    ["simworld-studio", "start",
     "--ue-host", "127.0.0.1",
     "--ue-port", str(SIMWORLD_PORT),
     "--port", "3002",
     "--data-dir", "/content/studio_workspace"],
    stdout=open("/content/studio.log", "w"),
    stderr=subprocess.STDOUT
)
time.sleep(8)

# Verify backend is up
import requests
try:
    r = requests.get("http://localhost:3002/api/health", timeout=10)
    print(f"  Arena backend running! Health: {r.json()}")
except Exception as e:
    print(f"  Arena backend may still be starting: {e}")
    print("  Check: !cat /content/studio.log")

print("\nAll services launched!")

## Cell 6: Get Your Browser URL

In [ ]:
"""Create a public tunnel URL to access the Arena in your browser."""
import subprocess, time, re, os, select

# Install cloudflared
print("Setting up tunnel...")
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        ["wget", "-q",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-O", "/usr/local/bin/cloudflared"],
        check=True
    )
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

# Kill any existing tunnel
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

# Start tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:3002", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait for URL — use select() for non-blocking reads
public_url = None
accumulated = ""
for _ in range(30):  # wait up to 30 seconds
    time.sleep(1)
    try:
        ready, _, _ = select.select([tunnel.stderr], [], [], 0)
        if ready:
            chunk = os.read(tunnel.stderr.fileno(), 8192).decode(errors='replace')
            accumulated += chunk
            match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', accumulated)
            if match:
                public_url = match.group(0)
                break
    except Exception:
        pass

if public_url:
    print(f"")
    print(f"{'='*60}")
    print(f"  SimWorld Studio is live!")
    print(f"")
    print(f"  Open in browser: {public_url}")
    print(f"{'='*60}")
    print(f"")
    print(f"Chat with Claude to build 3D urban scenes!")
    print(f"Try: 'Build a small neighborhood with 4 houses and trees'")
else:
    print("Tunnel failed to start. Trying Colab proxy fallback...")
    try:
        from google.colab.output import eval_js
        proxy_url = eval_js("google.colab.kernel.proxyPort(3002)")
        public_url = proxy_url
        print(f"Open in browser: {proxy_url}")
    except Exception:
        print("Both tunnel methods failed. Check your connection.")
        public_url = "http://localhost:3002"

## Cell 7: Verify Everything Works

Run this cell to confirm all services are healthy before using the Arena.

In [ ]:
"""Automated verification — all checks must pass before using the Arena."""
import requests, socket, subprocess, os

print("Running verification checks...\n")

results = {}

# 1. GPU
r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                   capture_output=True, text=True)
results["GPU"] = (r.returncode == 0, r.stdout.strip() if r.returncode == 0 else "Not found")

# 2. SimWorld binary
sw_path = globals().get('SIMWORLD_SH', '')
sw_ok = bool(sw_path) and os.path.exists(sw_path)
results["SimWorld Binary"] = (sw_ok, sw_path if sw_ok else "Not found")

# 3. SimWorld TCP
sw_port = globals().get('SIMWORLD_PORT', 9000)
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(5)
    s.connect(("127.0.0.1", sw_port))
    s.close()
    results[f"SimWorld TCP ({sw_port})"] = (True, "Connected")
except Exception as e:
    results[f"SimWorld TCP ({sw_port})"] = (False, str(e))

# 4. Arena Backend
try:
    r = requests.get("http://localhost:3002/api/health", timeout=10)
    results["Arena Backend"] = (r.status_code == 200, f"HTTP {r.status_code}")
except Exception as e:
    results["Arena Backend"] = (False, str(e))

# 5. Skills
try:
    r = requests.get("http://localhost:3002/api/skills", timeout=10)
    data = r.json()
    count = len(data) if isinstance(data, list) else 0
    results["Skills"] = (count > 0, f"{count} skills loaded")
except Exception as e:
    results["Skills"] = (False, str(e))

# 6. Assets
try:
    r = requests.get("http://localhost:3002/api/assets", timeout=10)
    results["Assets"] = (r.status_code == 200, "Catalog loaded")
except Exception as e:
    results["Assets"] = (False, str(e))

# 7. Claude Code CLI
r = subprocess.run(["claude", "--version"], capture_output=True, text=True)
results["Claude Code CLI"] = (r.returncode == 0,
    r.stdout.strip() if r.returncode == 0 else "Not installed")

# 8. API key
key_set = bool(os.environ.get("ANTHROPIC_API_KEY", ""))
results["API Key"] = (key_set, "Set" if key_set else "NOT SET")

# 9. Tunnel
tunnel_url = globals().get('public_url', '')
tunnel_ok = bool(tunnel_url) and tunnel_url.startswith("http")
results["Public URL"] = (tunnel_ok, tunnel_url if tunnel_ok else "Not created")

# Print results
all_ok = True
for name, (ok, detail) in results.items():
    icon = "[PASS]" if ok else "[FAIL]"
    if not ok:
        all_ok = False
    print(f"  {icon} {name}: {detail}")

print()
if all_ok:
    print("ALL CHECKS PASSED! SimWorld Studio is ready.")
    print(f"Open {tunnel_url} in your browser to start building 3D scenes.")
else:
    print("SOME CHECKS FAILED. Common fixes:")
    print("  - SimWorld TCP: wait 30-60s more, then re-run this cell")
    print("  - API key: run Cell 4")
    print("  - Tunnel: re-run Cell 6")
    print("  - Arena Backend: !cat /content/studio.log")

## Cell 8: Smoke Test (End-to-End)

Sends a test prompt through the full pipeline: **Chat -> Claude -> MCP Tools -> SimWorld**

In [ ]:
"""End-to-end smoke test — verifies the full pipeline works."""
import requests, json

print("Sending test prompt through the full pipeline...\n")
print("Prompt: 'Set up the environment with a sunny sky, then spawn one small building'\n")

try:
    response = requests.post("http://localhost:3002/api/chat", json={
        "message": "Set up the environment with a sunny sky, then spawn one small residential building (BP_Building_01) at the origin. Then take a screenshot.",
        "sessionId": None,
        "skills": ["building_placement"]
    }, stream=True, timeout=180)

    tool_calls = []
    text_output = []
    screenshots = []
    errors = []
    current_event = None  # Track SSE event type

    for line in response.iter_lines():
        if not line:
            continue
        decoded = line.decode('utf-8')

        # SSE comment lines (heartbeat)
        if decoded.startswith(':'):
            continue

        # Parse SSE event/data pairs
        if decoded.startswith('event: '):
            current_event = decoded[7:]
            continue
        if not decoded.startswith('data: '):
            continue

        try:
            data = json.loads(decoded[6:])
        except json.JSONDecodeError:
            continue

        event_type = current_event or 'unknown'

        if event_type == 'text':
            delta = data.get('delta', '')
            text_output.append(delta)
            print(delta, end='', flush=True)
        elif event_type == 'tool_start':
            name = data.get('displayName', data.get('name', '?'))
            tool_calls.append(name)
            print(f"\n  >> Tool: {name}", flush=True)
        elif event_type == 'tool_result':
            result = data.get('result', '')[:200]
            is_error = data.get('isError', False)
            if is_error:
                errors.append(result)
                print(f"\n  >> ERROR: {result}", flush=True)
            else:
                print(f"\n  >> Result: {result[:100]}...", flush=True)
        elif event_type == 'screenshot':
            screenshots.append(data.get('filepath', ''))
            print(f"\n  >> Screenshot captured!", flush=True)
        elif event_type == 'done':
            cost = data.get('costUsd')
            if cost:
                print(f"\n\n  Cost: ${cost:.4f}")

    print(f"\n\n{'='*50}")
    print(f"Smoke Test Results:")
    print(f"  Tool calls:  {len(tool_calls)} ({', '.join(tool_calls)})")
    print(f"  Screenshots: {len(screenshots)}")
    print(f"  Errors:      {len(errors)}")

    if len(tool_calls) > 0 and len(errors) == 0:
        print(f"\n  SMOKE TEST PASSED!")
        print(f"  Full pipeline working: Chat -> Claude -> MCP -> SimWorld")
    elif len(tool_calls) > 0:
        print(f"\n  PARTIAL PASS — tools fired but some errors occurred.")
    else:
        print(f"\n  SMOKE TEST FAILED — no tool calls detected.")
        print(f"  Check: API key set? SimWorld running? Logs: /content/studio.log")

except requests.exceptions.Timeout:
    print("\nSmoke test timed out (180s). Claude may be slow to respond.")
    print("The platform may still work — try using the browser UI.")
except Exception as e:
    print(f"\nSmoke test error: {e}")
    print("Check /content/studio.log for backend errors.")

---

## Troubleshooting

| Issue | Fix |
|---|---|
| "No GPU detected" | Runtime -> Change runtime type -> GPU (T4) |
| SimWorld TCP fails | Wait 60s more, re-run Cell 7 |
| Arena backend fails | Check: `!cat /content/studio.log` |
| Tunnel fails | Re-run Cell 6, or use Colab proxy |
| Claude errors | Verify API key in Cell 4 |
| Session expires | Colab free tier = 12h max. Re-run all cells. |

### View Logs
```python
!tail -50 /content/ue.log      # SimWorld logs
!tail -50 /content/studio.log    # Arena backend logs
```